# Transformer / self-attention from scratch

Every other notebook in this cookbook uses convolutions or plain MLPs (`huggingface/`'s
notebook uses transformers too, but only through a pretrained pipeline - a black box). Here we
implement multi-head self-attention from raw tensor ops and train it on a task chosen to make
the mechanism's necessity obvious: **sequence reversal**.

Given a random sequence of digits, predict it reversed. This needs no external data, has an
exact right answer (perfect for measuring accuracy precisely), and - crucially - is
*position-dependent*: you cannot solve it by looking at which digits are present, only by
knowing where each one is. That makes it a good demonstration of why transformers need
positional encoding, which we verify with an ablation at the end.

In [ ]:
import torch
import torch.nn as nn
import mlflow
import mlflow.pytorch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

mlflow.set_tracking_uri("file:///work/mlruns")
mlflow.set_experiment("transformer-ablation")

VOCAB = 10       # digits 0-9
SEQ_LEN = 8
LR = 1e-3
BATCH_SIZE = 64


def make_batch(batch):
    x = torch.randint(0, VOCAB, (batch, SEQ_LEN), device=device)
    y = x.flip(dims=[1])  # the task: reverse the sequence
    return x, y

## Multi-head self-attention, from raw tensor ops

For every position, project the input into query/key/value vectors. Each position's query is
compared (dot product) against every position's key to get attention weights - how much that
position should "listen to" every other position - then those weights combine the value
vectors into the output. Splitting into multiple heads lets the model attend to different kinds
of relationships in parallel.

Nothing here is transformer-library magic - it's `reshape`, `matmul`, and `softmax`.

In [ ]:
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.out = nn.Linear(d_model, d_model)

    def forward(self, x):
        B, L, D = x.shape
        qkv = self.qkv(x).reshape(B, L, 3, self.n_heads, self.d_head).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]  # each (B, n_heads, L, d_head)
        scores = (q @ k.transpose(-2, -1)) / (self.d_head ** 0.5)  # (B, n_heads, L, L)
        attn = torch.softmax(scores, dim=-1)
        out = (attn @ v).transpose(1, 2).reshape(B, L, D)
        return self.out(out)


class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff):
        super().__init__()
        self.attn = MultiHeadSelfAttention(d_model, n_heads)
        self.norm1 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model))
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x):
        x = x + self.attn(self.norm1(x))   # pre-norm + residual, as in modern transformers
        x = x + self.ff(self.norm2(x))
        return x


class TinyTransformer(nn.Module):
    def __init__(self, vocab_size, seq_len, d_model=64, n_heads=4, n_layers=2, d_ff=128, use_pos=True):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.use_pos = use_pos
        if use_pos:
            self.pos_emb = nn.Embedding(seq_len, d_model)
        self.blocks = nn.ModuleList([TransformerBlock(d_model, n_heads, d_ff) for _ in range(n_layers)])
        self.head = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        B, L = x.shape
        h = self.tok_emb(x)
        if self.use_pos:
            pos = torch.arange(L, device=x.device).unsqueeze(0)
            h = h + self.pos_emb(pos)
        for block in self.blocks:
            h = block(h)
        return self.head(h)  # (B, L, vocab_size) logits per position

In [ ]:
from torch.utils.tensorboard import SummaryWriter


def train(use_pos, steps=1201, log_dir=None):
    torch.manual_seed(0)
    model = TinyTransformer(VOCAB, SEQ_LEN, use_pos=use_pos).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    writer = SummaryWriter(log_dir=log_dir) if log_dir else None
    run_name = "with-positions" if use_pos else "without-positions"

    with mlflow.start_run(run_name=run_name) as run:
        mlflow.log_params({
            "use_positional_encoding": use_pos,
            "steps": steps,
            "batch_size": BATCH_SIZE,
            "learning_rate": LR,
            "vocab_size": VOCAB,
            "sequence_length": SEQ_LEN,
        })

        for step in range(steps):
            x, y = make_batch(BATCH_SIZE)
            opt.zero_grad()
            logits = model(x)
            loss = nn.functional.cross_entropy(logits.reshape(-1, VOCAB), y.reshape(-1))
            loss.backward()
            opt.step()
            if writer:
                writer.add_scalar("loss/train", loss.item(), step)
            mlflow.log_metric("loss_train", loss.item(), step=step)
            if step % 300 == 0:
                print(f"  step {step:5d}  loss {loss.item():.4f}")

        if writer:
            writer.close()
        run_id = run.info.run_id
    return model, run_id


def evaluate(model, run_id=None):
    x, y = make_batch(512)
    with torch.no_grad():
        pred = model(x).argmax(dim=-1)
        token_acc = (pred == y).float().mean().item()
        seq_acc = (pred == y).all(dim=1).float().mean().item()
    if run_id:
        with mlflow.start_run(run_id=run_id):
            mlflow.log_metric("token_accuracy", token_acc)
            mlflow.log_metric("sequence_accuracy", seq_acc)
            mlflow.pytorch.log_model(model, "model")
    return token_acc, seq_acc


print("training with positional encoding...")
model, run_id = train(use_pos=True, log_dir="/work/runs/transformer/with-positions")
token_acc, seq_acc = evaluate(model, run_id)
print(f"held-out: per-token accuracy {token_acc:.3f}, exact-sequence accuracy {seq_acc:.3f}")

## Ablation: does positional encoding actually matter?

Without it, self-attention is *permutation-equivariant* - every position gets treated
identically regardless of where it sits in the sequence, so the model has no way to know
"reverse" means position 0 goes to the last slot. Train the exact same architecture with
`use_pos=False` and compare.

In [ ]:
print("training WITHOUT positional encoding (ablation)...")
model_no_pos, run_id_no_pos = train(use_pos=False, log_dir="/work/runs/transformer/without-positions")
token_acc_no_pos, seq_acc_no_pos = evaluate(model_no_pos, run_id_no_pos)

print()
print(f"{'':20s}{'token acc':>12s}{'exact-seq acc':>16s}")
print(f"{'with positions':20s}{token_acc:12.3f}{seq_acc:16.3f}")
print(f"{'without positions':20s}{token_acc_no_pos:12.3f}{seq_acc_no_pos:16.3f}")

## Next steps

- Swap the task for sorting instead of reversal - harder, since the output/input alignment
  isn't a fixed permutation, and see how much longer it takes to converge.
- Make it autoregressive: add a causal mask (`scores.masked_fill_(causal_mask, -inf)` before
  the softmax in `MultiHeadSelfAttention`) and train next-token prediction instead of
  predicting the whole reversed sequence at once - this is the shape real language models use.
- Try `huggingface/notebook.ipynb` to see the same underlying architecture at production scale,
  pretrained rather than trained from scratch here.
- Log attention weights (`attn` inside `MultiHeadSelfAttention.forward`) to TensorBoard as an
  image and watch which positions the model learns to attend to.